<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/LotteryTicketSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch, os, gc
import torch.nn.utils.prune as prune
from transformers import AutoTokenizer, AutoModelForCausalLM

# Speicherpfad und Device
model_name = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
device = "cuda" if torch.cuda.is_available() else "cpu"
init_path = "./init_weights"

# Lade Modell & Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True
)

model.eval()
model = torch.compile(model)
model.to(device)

# Hilfsfunktion: prunebare Keys (Linear-Layer) extrahieren
def get_prunable_keys(model):
    return [
        k for k in model.state_dict().keys()
        if 'weight' in k and any(x in k for x in ['mlp', 'self_attn', 'attn', 'linear', 'dense'])
    ]

# Initialgewichte speichern (nur prunebare Layer, FP16, auf Disk)
def save_init_weights(model, keys, path=init_path):
    os.makedirs(path, exist_ok=True)
    for k in keys:
        tensor = model.state_dict()[k].half().cpu()
        torch.save(tensor, os.path.join(path, f"{k.replace('.', '_')}.pt"))

def load_init_weight(key, path=init_path):
    return torch.load(os.path.join(path, f"{key.replace('.', '_')}.pt")).float()

def prune_linear_layers_cpu(model, amount):
    device = next(model.parameters()).device
    model.cpu()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=amount)
    model.to(device)
    torch.cuda.empty_cache()  # Speicher aufräumen
    gc.collect()  # Garbage Collector aufrufen


def reset_weights_to_init(model, keys, path='./init_weights'):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and hasattr(module, 'weight_orig') and hasattr(module, 'weight_mask'):
            param_key = f"{name}.weight_orig"
            if param_key in keys:
                orig = torch.load(os.path.join(path, f"{param_key.replace('.', '_')}.pt")).to(module.weight_orig.device)
                mask = module.weight_mask
                with torch.no_grad():
                    module.weight_orig.data = module.weight_orig.data * (1-mask) + orig * mask

def remove_pruning(model):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            # Nur wenn Mask/Orig vorhanden
            if hasattr(module, "weight_mask") and hasattr(module, "weight_orig"):
                prune.remove(module, "weight")

prunable_keys = get_prunable_keys(model)
save_init_weights(model, prunable_keys, path=init_path)

n_iter = 5
prune_each = 1 - (1-0.5)**(1/5)    # ≈ 0.13, damit 5 Iterationen zusammen 50%

for i in range(n_iter):
    print(f"=== LTH Iteration {i+1}/{n_iter} | Prune {prune_each*100:.2f}% ===")
    prune_linear_layers_cpu(model, amount=prune_each)
    reset_weights_to_init(model, prunable_keys, path=init_path)
    remove_pruning(model)

# Originale Maske entfernen
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if hasattr(module, "weight_mask") and hasattr(module, "weight_orig"):
            prune.remove(module, "weight")

output_dir = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH"

# PyTorch-Checkpoint
model.save_pretrained(output_dir, safe_serialization=True)

# Tokenizer speichern
tokenizer.save_pretrained(output_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

=== LTH Iteration 1/5 | Prune 12.94% ===
=== LTH Iteration 2/5 | Prune 12.94% ===
=== LTH Iteration 3/5 | Prune 12.94% ===
=== LTH Iteration 4/5 | Prune 12.94% ===
=== LTH Iteration 5/5 | Prune 12.94% ===


('/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/vocab.json',
 '/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/merges.txt',
 '/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH/tokenizer.json')

In [12]:
# ==== 10. Metriken zeigen ====
def compute_pruned_stats(model):
    total_params = 0
    nonzero_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dim() > 1 and "weight" in name:
            total_params += param.numel()
            nonzero_params += torch.count_nonzero(param).item()
    zero_params = total_params - nonzero_params
    sparsity = 100.0 * zero_params / total_params
    compression_ratio = total_params / nonzero_params if nonzero_params > 0 else float("inf")
    print(f"Gesamtparameter: {total_params:,}")
    print(f"Aktive Parameter (<> 0): {nonzero_params:,}")
    print(f"Sparsity: {sparsity:.2f}%")
    print(f"Komprimierungsrate: {compression_ratio:.2f}x")
    return total_params, nonzero_params, sparsity, compression_ratio

compute_pruned_stats(model)

Gesamtparameter: 4,022,272,000
Aktive Parameter (<> 0): 2,011,136,000
Sparsity: 50.00%
Komprimierungsrate: 2.00x


(4022272000, 2011136000, 50.0, 2.0)